In [1]:
import os
import yaml
import pandas as pd
import glob

def extract_and_transform(input_dir, output_dir):
    all_data = []
    
    # Path pattern: input_dir/Month_Folder/*.yaml
    yaml_files = glob.glob(os.path.join(input_dir, '**/*.yaml'), recursive=True)
    
    for file_path in yaml_files:
        with open(file_path, 'r') as f:
            content = yaml.safe_load(f)
            # Content expected to be a list of dicts: [{symbol: 'AAPL', close: 150, ...}, ...]
            # Adding date from filename or folder if necessary
            date_str = os.path.splitext(os.path.basename(file_path))[0]
            for entry in content:
                entry['Date'] = date_str
                all_data.append(entry)

    df = pd.DataFrame(all_data)
    df['Date'] = pd.to_datetime(df['Date'])
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Group by Symbol and save 50 CSVs
    symbols = df['Symbol'].unique()
    for symbol in symbols:
        symbol_df = df[df['Symbol'] == symbol].sort_values('Date')
        symbol_df.to_csv(f"{output_dir}/{symbol}.csv", index=False)
    
    return df

# Usage
# master_df = extract_and_transform('raw_yaml_data', 'symbol_csvs')

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def run_stock_analysis(df, sector_df):
    # --- Data Prep ---
    df = df.sort_values(['Symbol', 'Date'])
    df['Daily_Return'] = df.groupby('Symbol')['Close'].pct_change()
    
    # Yearly metrics per stock
    yearly_stats = df.groupby('Symbol').agg({
        'Close': ['first', 'last', 'mean'],
        'Volume': 'mean',
        'Daily_Return': 'std'
    })
    yearly_stats.columns = ['Start_Price', 'End_Price', 'Avg_Price', 'Avg_Volume', 'Volatility']
    yearly_stats['Yearly_Return'] = (yearly_stats['End_Price'] - yearly_stats['Start_Price']) / yearly_stats['Start_Price']

    # --- Market Summary ---
    top_10_green = yearly_stats.nlargest(10, 'Yearly_Return')
    top_10_loss = yearly_stats.nsmallest(10, 'Yearly_Return')
    green_count = len(yearly_stats[yearly_stats['Yearly_Return'] > 0])
    red_count = len(yearly_stats[yearly_stats['Yearly_Return'] <= 0])

    print(f"Market Summary: {green_count} Green, {red_count} Red")
    print(f"Global Avg Price: {yearly_stats['Avg_Price'].mean():.2f}")

    # 1. Volatility Analysis
    plt.figure(figsize=(10, 6))
    top_volatility = yearly_stats.nlargest(10, 'Volatility')
    sns.barplot(x=top_volatility.index, y=top_volatility['Volatility'], palette='magma')
    plt.title('Top 10 Most Volatile Stocks')
    plt.show()

    # 2. Cumulative Return
    df['Cum_Return'] = df.groupby('Symbol')['Daily_Return'].transform(lambda x: (1 + x).cumprod() - 1)
    top_5_symbols = yearly_stats.nlargest(5, 'Yearly_Return').index
    
    plt.figure(figsize=(12, 6))
    for sym in top_5_symbols:
        subset = df[df['Symbol'] == sym]
        plt.plot(subset['Date'], subset['Cum_Return'], label=sym)
    plt.title('Cumulative Return: Top 5 Performers')
    plt.legend()
    plt.show()

    # 3. Sector-wise Performance
    # Merge with sector_df (columns: Symbol, Sector)
    merged_stats = yearly_stats.reset_index().merge(sector_df, on='Symbol')
    sector_perf = merged_stats.groupby('Sector')['Yearly_Return'].mean().sort_values()
    
    plt.figure(figsize=(10, 6))
    sector_perf.plot(kind='barh', color='skyblue')
    plt.title('Average Yearly Return by Sector')
    plt.show()

    # 4. Stock Price Correlation
    pivot_df = df.pivot(index='Date', columns='Symbol', values='Close')
    corr_matrix = pivot_df.pct_change().corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0)
    plt.title('Stock Price Correlation Heatmap')
    plt.show()

    # 5. Month-wise Gainers/Losers
    df['Month'] = df['Date'].dt.to_period('M')
    # Calculate monthly return as (Last Close of Month / First Close of Month) - 1
    monthly_data = df.groupby(['Symbol', 'Month'])['Close'].agg(['first', 'last'])
    monthly_data['Monthly_Return'] = (monthly_data['last'] - monthly_data['first']) / monthly_data['first']
    
    months = df['Month'].unique()
    fig, axes = plt.subplots(4, 3, figsize=(20, 15)) # 12 Months
    axes = axes.flatten()

    for i, month in enumerate(sorted(months)):
        m_df = monthly_data.xs(month, level='Month')
        top_m = m_df.nlargest(5, 'Monthly_Return')
        bot_m = m_df.nsmallest(5, 'Monthly_Return')
        combined = pd.concat([top_m, bot_m]).sort_values('Monthly_Return')
        
        colors = ['red' if x < 0 else 'green' for x in combined['Monthly_Return']]
        combined['Monthly_Return'].plot(kind='bar', ax=axes[i], color=colors)
        axes[i].set_title(f'Top Gainers/Losers: {month}')
        axes[i].set_xticklabels(combined.index, rotation=45)

    plt.tight_layout()
    plt.show()

# Example execution flow:
# sector_info = pd.read_csv('sectors.csv')
# run_stock_analysis(master_df, sector_info)